# Week 8 Exercise - Agentic Deal Hunter (No Framework)

This notebook builds the Week 8 system **without LangGraph or DealAgentFramework**.

Pipeline:
1. Scan deals
2. Estimate true value using available pricing agents
3. Score discount opportunities
4. Select best deal
5. Optionally notify user
6. Show everything in Gradio

In [ ]:
# Install once in notebook environments if needed
!pip -q install -U python-decouple

In [ ]:
import os
import sys
import time
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import gradio as gr
from dotenv import load_dotenv
from decouple import config

load_dotenv(override=True)

# OpenRouter wiring (agents use OpenAI() internally, so set OpenAI env vars)
OPENROUTER_API_KEY = config("OPEN_ROUTER_API_KEY", default=config("OPENROUTER_API_KEY", default=None))
OPENROUTER_BASE_URL = config("OPENROUTER_BASE_URL", default="https://openrouter.ai/api/v1")

if OPENROUTER_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENROUTER_API_KEY
    os.environ["OPENAI_BASE_URL"] = OPENROUTER_BASE_URL
    print("OpenRouter configured via OPENAI_* env vars")
else:
    print("OpenRouter key not found. Set OPEN_ROUTER_API_KEY (or OPENROUTER_API_KEY).")

SCANNER_MODEL = config("OPENROUTER_SCANNER_MODEL", default="openai/gpt-4o-mini")
FRONTIER_MODEL = config("OPENROUTER_FRONTIER_MODEL", default="openai/gpt-4o-mini")

# Resolve week8 module path so imports work from community notebook location
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent, cwd.parent.parent.parent]
WEEK8_DIR = None
for c in candidates:
    if (c / "week8").exists():
        WEEK8_DIR = c / "week8"
        break
if WEEK8_DIR is None and (cwd / "agents").exists():
    WEEK8_DIR = cwd
if WEEK8_DIR is None:
    raise RuntimeError("Could not locate week8 directory for imports.")

if str(WEEK8_DIR) not in sys.path:
    sys.path.insert(0, str(WEEK8_DIR))

print("Using WEEK8_DIR:", WEEK8_DIR)
print("Scanner model:", SCANNER_MODEL)
print("Frontier model:", FRONTIER_MODEL)

In [ ]:
import chromadb
from agents.scanner_agent import ScannerAgent
from agents.messaging_agent import MessagingAgent
from agents.frontier_agent import FrontierAgent
from agents.specialist_agent import SpecialistAgent
from agents.neural_network_agent import NeuralNetworkAgent

# Core runtime settings
DEAL_THRESHOLD = 50.0
MAX_CANDIDATES = 5
WEIGHTS = {
    "frontier": 0.80,
    "specialist": 0.10,
    "neural": 0.10,
}

# Agent toggles (turn off if infra not configured)
ENABLE_FRONTIER = True
ENABLE_SPECIALIST = False  # needs Modal service configured
ENABLE_NEURAL = False      # needs deep_neural_network.pth available
ENABLE_NOTIFY = False      # needs PUSHOVER env vars

DB_PATH = str(WEEK8_DIR / "products_vectorstore")
DB_COLLECTION = "products"

In [ ]:
RUNTIME: Dict = {
    "collection": None,
    "scanner": None,
    "messenger": None,
    "frontier": None,
    "specialist": None,
    "neural": None,
}


def get_collection(logs: List[str]):
    if RUNTIME["collection"] is not None:
        return RUNTIME["collection"]
    try:
        client = chromadb.PersistentClient(path=DB_PATH)
        collection = client.get_or_create_collection(DB_COLLECTION)
        RUNTIME["collection"] = collection
        logs.append(f"Connected to Chroma at {DB_PATH} collection={DB_COLLECTION}")
    except Exception as e:
        logs.append(f"Chroma unavailable: {e}")
        RUNTIME["collection"] = None
    return RUNTIME["collection"]


def get_scanner(logs: List[str]):
    if RUNTIME["scanner"] is None:
        RUNTIME["scanner"] = ScannerAgent()
        RUNTIME["scanner"].MODEL = SCANNER_MODEL
        logs.append(f"ScannerAgent ready (model={RUNTIME['scanner'].MODEL})")
    return RUNTIME["scanner"]


def get_messenger(logs: List[str]):
    if RUNTIME["messenger"] is None:
        try:
            RUNTIME["messenger"] = MessagingAgent()
            logs.append("MessagingAgent ready")
        except Exception as e:
            logs.append(f"MessagingAgent unavailable: {e}")
            RUNTIME["messenger"] = None
    return RUNTIME["messenger"]


def get_frontier(logs: List[str]):
    if not ENABLE_FRONTIER:
        return None
    if RUNTIME["frontier"] is None:
        collection = get_collection(logs)
        if collection is None:
            return None
        try:
            RUNTIME["frontier"] = FrontierAgent(collection)
            RUNTIME["frontier"].MODEL = FRONTIER_MODEL
            logs.append(f"FrontierAgent ready (model={RUNTIME['frontier'].MODEL})")
        except Exception as e:
            logs.append(f"FrontierAgent unavailable: {e}")
            RUNTIME["frontier"] = None
    return RUNTIME["frontier"]


def get_specialist(logs: List[str]):
    if not ENABLE_SPECIALIST:
        return None
    if RUNTIME["specialist"] is None:
        try:
            RUNTIME["specialist"] = SpecialistAgent()
            logs.append("SpecialistAgent ready")
        except Exception as e:
            logs.append(f"SpecialistAgent unavailable: {e}")
            RUNTIME["specialist"] = None
    return RUNTIME["specialist"]


def get_neural(logs: List[str]):
    if not ENABLE_NEURAL:
        return None
    if RUNTIME["neural"] is None:
        try:
            RUNTIME["neural"] = NeuralNetworkAgent()
            logs.append("NeuralNetworkAgent ready")
        except Exception as e:
            logs.append(f"NeuralNetworkAgent unavailable: {e}")
            RUNTIME["neural"] = None
    return RUNTIME["neural"]

In [ ]:
def estimate_with_agents(description: str, logs: List[str]) -> Dict[str, float]:
    preds = {}

    frontier = get_frontier(logs)
    if frontier is not None:
        try:
            preds["frontier"] = float(frontier.price(description))
        except Exception as e:
            logs.append(f"Frontier price error: {e}")

    specialist = get_specialist(logs)
    if specialist is not None:
        try:
            preds["specialist"] = float(specialist.price(description))
        except Exception as e:
            logs.append(f"Specialist price error: {e}")

    neural = get_neural(logs)
    if neural is not None:
        try:
            preds["neural"] = float(neural.price(description))
        except Exception as e:
            logs.append(f"Neural price error: {e}")

    return preds


def weighted_estimate(preds: Dict[str, float]) -> Optional[float]:
    if not preds:
        return None
    active_weights = {k: WEIGHTS.get(k, 0.0) for k in preds.keys()}
    total = sum(active_weights.values())
    if total <= 0:
        return float(np.mean(list(preds.values())))
    return sum(preds[k] * active_weights[k] for k in preds.keys()) / total


def to_table(memory_records: List[Dict]) -> pd.DataFrame:
    if not memory_records:
        return pd.DataFrame(columns=["description", "price", "estimate", "discount", "url", "methods"])
    rows = []
    for r in memory_records:
        rows.append({
            "description": r["description"][:120],
            "price": round(r["price"], 2),
            "estimate": round(r["estimate"], 2),
            "discount": round(r["discount"], 2),
            "url": r["url"],
            "methods": ", ".join(sorted(r.get("methods", []))),
        })
    return pd.DataFrame(rows).sort_values("discount", ascending=False)


def run_cycle(memory_records: List[Dict], threshold: float, max_candidates: int, do_notify: bool):
    logs: List[str] = []
    scanner = get_scanner(logs)

    seen_urls = {x["url"] for x in memory_records}
    selection = scanner.scan(memory=[])
    if selection is None or not selection.deals:
        logs.append("No deals returned by scanner")
        return memory_records, to_table(memory_records), "No new deals found", "\n".join(logs)

    new_records = []
    deals = [d for d in selection.deals if d.url not in seen_urls][:max_candidates]
    logs.append(f"Scanner produced {len(selection.deals)} deals; {len(deals)} are new")

    for deal in deals:
        preds = estimate_with_agents(deal.product_description, logs)
        estimate = weighted_estimate(preds)
        if estimate is None:
            logs.append(f"Skipping deal (no estimator output): {deal.url}")
            continue

        discount = float(estimate - float(deal.price))
        new_records.append({
            "description": deal.product_description,
            "price": float(deal.price),
            "estimate": float(estimate),
            "discount": discount,
            "url": deal.url,
            "methods": list(preds.keys()),
        })

    if not new_records:
        logs.append("No scored opportunities in this run")
        return memory_records, to_table(memory_records), "No scored opportunities", "\n".join(logs)

    memory_records = memory_records + new_records
    table = to_table(memory_records)

    best = max(new_records, key=lambda x: x["discount"])
    status = f"Best new opportunity discount: ${best['discount']:.2f}"

    if do_notify and ENABLE_NOTIFY and best["discount"] >= threshold:
        messenger = get_messenger(logs)
        if messenger is not None:
            try:
                messenger.notify(best["description"], best["price"], best["estimate"], best["url"])
                logs.append("Notification sent")
            except Exception as e:
                logs.append(f"Notification failed: {e}")

    return memory_records, table, status, "\n".join(logs)

In [ ]:
# quick dry run (without UI)
memory = []
memory, table, status, run_logs = run_cycle(memory, threshold=DEAL_THRESHOLD, max_candidates=MAX_CANDIDATES, do_notify=False)
print(status)
print(run_logs)
display(table.head(10))

In [ ]:
with gr.Blocks(title="Week 8 - Agentic Deal Hunter") as demo:
    gr.Markdown("## Week 8 Agentic Deal Hunter (No Framework)")
    gr.Markdown("Scan -> Value -> Score -> Select -> Notify")

    memory_state = gr.State([])

    with gr.Tab("Run Controls"):
        threshold_in = gr.Slider(0, 500, value=DEAL_THRESHOLD, step=5, label="Minimum discount for notify")
        max_candidates_in = gr.Slider(1, 10, value=MAX_CANDIDATES, step=1, label="Max new candidates per scan")
        notify_in = gr.Checkbox(value=False, label="Send notification when threshold met")

        run_btn = gr.Button("Run One Agent Cycle")
        status_out = gr.Textbox(label="Status")

    with gr.Tab("Live Deals"):
        deals_out = gr.Dataframe(label="Opportunities", wrap=True)

    with gr.Tab("Agent Trace"):
        logs_out = gr.Textbox(label="Run Logs", lines=18)

    def on_run(memory_records, threshold, max_candidates, do_notify):
        return run_cycle(memory_records, threshold, int(max_candidates), do_notify)

    run_btn.click(
        fn=on_run,
        inputs=[memory_state, threshold_in, max_candidates_in, notify_in],
        outputs=[memory_state, deals_out, status_out, logs_out],
    )

demo.launch()